## Versión final organizada

Esta copia escribe sus salidas bajo `outputs/`. Las rutas comunes se definen en `project_paths.py`.


In [ ]:
from project_paths import ANALYSIS_DATA_DIR, CLE_NATURE_DATA_DIR, CLE_NATURE_FIGURES_DIR, ensure_output_directories
ensure_output_directories()


## Computing Conditional Lyapunov Exponents for NATURE Run (NATURE-CLE)

##### Loading Python Modules:

In [ ]:
import numpy as np
import pickle
import random
import sys
import math
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from class_Lorenz63 import run_RK4
from class_state_vec import state_vector
from class_obs import obs_da
from class_da_sys import da_system
from class_cse import cse

#### Defining lorenz63 model:

In [ ]:
sigma = 10.0       # Parameter sigma
rho   = 28.0       # Parameter rho
beta  = 8.0 / 3.0  # Parameter beta
dt    = 0.01       # Integration time step

def lorenz63(state, t):
        """
        Calculates the derivatives for the Lorenz63 system.
        
        Parameters:
        state: vector [x, y, z] representing the current state.
        t: current time (not used in Lorenz63 but included for compatibility).
        
        Returns:
        A vector of derivatives [dx/dt, dy/dt, dz/dt].
        """
        x, y, z = state  # Unpack the state vector
        dx = sigma * (y - x)              # Equation for dx/dt
        dy = x * (rho - z) - y            # Equation for dy/dt
        dz = x * y - beta * z             # Equation for dz/dt
        return np.array([dx, dy, dz])     # Return the derivative vector

In [ ]:
PathFigures = str(CLE_NATURE_FIGURES_DIR)
#sv.setFigurePath(PathFigures)

#### Load NATURE Run:

In [ ]:
#-----------------------------------------------------------
# Reading da system object:
#-----------------------------------------------------------
name     = 'x_analysis'
infile = str(ANALYSIS_DATA_DIR / (name + '_init.pkl'))
das      = da_system()
das      = das.load(infile)
sv       = das.getStateVector()
x_nature = sv.getTrajectory()
print('x_nature: ', x_nature)
#print(das)

#### Plot Nature run:

In [ ]:
t0 = 149517 # Starting time used to initialize nature run in the CSE 
Ta=8
num_cycles=300

# Obain the portion of the nature run to plot:
x_nature[:,t0:t0+Ta*num_cycles]

# Create simple figure
fig, ax = plt.subplots(figsize=(14, 6))

# Plot Nature Run (reference truth)
ax.plot(np.arange(3000), x_nature[0, t0:t0 + 3000], 'k-', linewidth=2.5, 
        label='Nature Run', alpha=0.8, zorder=10)

# Add zero reference line
ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.3)

ax.axvline(x=0, color='red', linestyle = '--')
ax.axvline(x=300, color='red', linestyle = '--')


# Labels and formatting
ax.set_xlabel('Time Steps', fontsize=12)
ax.set_ylabel('X Component', fontsize=12)
#ax.set_title(f'X Component: Ensemble Members vs Nature Run (first {max_steps} steps)', 
#             fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='best', ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
#plt.savefig(PathFigures + 'ensemble_vs_nature_X.png', dpi=300, bbox_inches='tight')
plt.show()

## STEP 3: Ensemble-Based Lyapunov Exponent Calculation

**Objective**: Compute the leading Lyapunov exponent using a robust ensemble-based approach that samples multiple attractor points and perturbations.

**Methodology**: 
1. Select multiple random points on the attractor
2. Apply multiple random perturbations to each point
3. Track the growth of perturbation distances over time
4. Compute the Lyapunov exponent from the **ensemble mean distance** (not average of individual fits)

**Why this method**: Averaging over many attractor points and perturbations reduces dependence on initial conditions and provides a statistically robust estimate of the leading Lyapunov exponent.

In [ ]:
# ============================================================================
# STEP 3: CONFIGURATION
# ============================================================================

# Ensemble parameters
n_attractor_points_LE = 400      # Number of random attractor points to sample
n_perturbations_LE = 100         # Number of perturbations per attractor point
total_trajectories_LE = n_attractor_points_LE * n_perturbations_LE  # Total: 40,000

# Evolution parameters
n_steps_LE = 3500                # Number of time steps to evolve (35 seconds with dt=0.01)
perturbation_std_LE = 1e-8       # Perturbation magnitude (same as STEP 1 and 2)

# Lyapunov exponent fitting window
FIT_END_LE = 1500                # Fit window (up to step 1500)

print(f"{'='*70}")
print(f"  STEP 3: Ensemble-Based Lyapunov Exponent Configuration")
print(f"{'='*70}")
print(f"\nEnsemble parameters:")
print(f"  - Attractor points: {n_attractor_points_LE}")
print(f"  - Perturbations per point: {n_perturbations_LE}")
print(f"  - Total trajectories: {total_trajectories_LE:,}")
print(f"\nEvolution parameters:")
print(f"  - Time steps: {n_steps_LE}")
#print(f"  - Integration time: {n_steps_LE * lorenz_stepper.dt:.1f} seconds")
#print(f"  - dt = {lorenz_stepper.dt}")
print(f"  - Perturbation magnitude: {perturbation_std_LE}")
print(f"\nFitting parameters:")
print(f"  - FIT_END: {FIT_END_LE} steps")
#print(f"  - Fit window: 0 to {FIT_END_LE * lorenz_stepper.dt:.1f} seconds")
print(f"  - Reference λ (Viswanath 1998): 0.9056")

In [ ]:
# Use the nature run as the long trajectory for selection
# Transpose x_nature from (3, time_steps) to (time_steps, 3)
long_trj_for_selection = x_nature.T[0:8000,:]

print(f"✓ Nature run prepared for CLE calculation")
print(f"  Original x_nature shape: {x_nature.shape}")
print(f"  long_trj_for_selection shape: {long_trj_for_selection.shape}")
print(f"  Total time steps available: {long_trj_for_selection.shape[0]:,}")

In [ ]:
# ============================================================================
# SELECT RANDOM ATTRACTOR POINTS
# ============================================================================

# Use the existing long trajectory to sample random attractor points
np.random.seed(42)  # For reproducibility

# Sample random indices from the long trajectory (avoiding early transients)
random_indices_LE = np.random.choice(
    np.arange(50, len(long_trj_for_selection)), 
    size=n_attractor_points_LE, 
    replace=False
)

# Get the attractor points
attractor_points_LE = long_trj_for_selection[random_indices_LE]

print(f"\n{'='*70}")
print(f"  Attractor Point Selection")
print(f"{'='*70}")
print(f"\nSelected {n_attractor_points_LE} random attractor points")
print(f"  Shape: {attractor_points_LE.shape}")
print(f"  Source: long_trj_for_selection (avoiding first 5000 steps)")
print(f"\nExample attractor points (first 3):")
for i in range(min(3, n_attractor_points_LE)):
    pt = attractor_points_LE[i]
    print(f"  Point {i+1}: [{pt[0]:7.3f}, {pt[1]:7.3f}, {pt[2]:7.3f}]")

In [ ]:
# ============================================================================
# FIGURE 14: VISUALIZE SELECTED ATTRACTOR POINTS IN 3D
# ============================================================================

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot the full trajectory in gray for context
ax.plot(long_trj_for_selection[:, 0], long_trj_for_selection[:, 1], long_trj_for_selection[:, 2], 
        linewidth=0.3, alpha=0.6, color='green', label='Full trajectory')

# Plot the selected attractor points in red
ax.scatter(attractor_points_LE[:, 0], attractor_points_LE[:, 1], attractor_points_LE[:, 2],
          color='red', s=100, marker='o', edgecolors='darkred', linewidths=2,
          label=f'Selected points (n={n_attractor_points_LE})', zorder=5)

ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('Y', fontsize=12)
ax.set_zlabel('Z', fontsize=12)
ax.set_title(f'Lorenz Attractor with {n_attractor_points_LE} Selected Points for Ensemble', 
            fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PathFigures}/fig1_attractor_points_n{n_attractor_points_LE}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Figure 1 saved: fig1_attractor_points_n{n_attractor_points_LE}.png")

In [ ]:
# ============================================================================
# DEFINE TRAJECTORY PRODUCTION FUNCTION
# ============================================================================

def produce_trj(initial_state, n_steps):
    """
    Produce a trajectory by integrating the Lorenz63 system.
    
    Parameters:
    -----------
    initial_state : array-like, shape (3,)
        Initial condition [x, y, z]
    n_steps : int
        Number of time steps to integrate
    
    Returns:
    --------
    trajectory : ndarray, shape (n_steps+1, 3)
        Trajectory including initial state
    """
    t_final = n_steps * dt  # Uses global dt parameter
    
    # Create RK4 solver
    solver = run_RK4(lorenz63, sigma, beta, rho, initial_state, dt, t_final)
    
    # Run integration using the same approach as in gen_ana_ens.ipynb
    trajectory = solver.integrate()
    
    return trajectory

print(f"✓ Function produce_trj defined")
print(f"  Uses run_RK4 with .integrate() method")
print(f"  Time step: dt = {dt} (global parameter)")
print(f"  Output shape: (n_steps+1, 3)")

### Computing Ensemble Distances (with caching)

**Note:** The ensemble distance computation takes approximately **24 minutes** to complete. 

To avoid recomputing every time:
- Results are automatically **saved** to `distances_ensemble_nature_n{points}_npert{perts}.pkl` after computation
- On subsequent runs, results are **loaded** from cache instantly
- To force recomputation, simply delete the cache file and re-run
- Cache filename includes configuration (n_attractor_points and n_perturbations) to avoid conflicts

In [ ]:
# ============================================================================
# TRY TO LOAD PRE-COMPUTED ENSEMBLE DISTANCES (saves ~24 minutes)
# ============================================================================

# Generate cache filename based on configuration
cache_filename = CLE_NATURE_DATA_DIR / f'distances_ensemble_nature_n{n_attractor_points_LE}_npert{n_perturbations_LE}.pkl'

try:
    with open(cache_filename, 'rb') as f:
        distances_ensemble = pickle.load(f)
    
    print("="*70)
    print("  ✓ PRE-COMPUTED ENSEMBLE DISTANCES LOADED FROM CACHE")
    print("="*70)
    print(f"\n✓ File loaded: {cache_filename}")
    print(f"✓ distances_ensemble shape: {distances_ensemble.shape}")
    print(f"  (attractor_points, perturbations, time_steps) = {distances_ensemble.shape}")
    print(f"\n⚠️  Skipping computation (using cached results)")
    print(f"   To recompute, delete '{cache_filename}' and re-run")
    
    # Set flag to skip computation
    skip_ensemble_computation = True
    
except FileNotFoundError:
    print("="*70)
    print("  ⚠️  No cached ensemble distances found")
    print("="*70)
    print(f"\n  Cache file not found: {cache_filename}")
    print(f"  Will compute ensemble distances (this takes ~24 minutes)")
    print(f"  Results will be saved to '{cache_filename}'")
    
    # Set flag to run computation
    skip_ensemble_computation = False

### Visualizing Perturbation Evolution

Before computing the full ensemble, let's visualize how perturbations evolve in the NATURE trajectory:
- Select **1 random point** from the NATURE attractor
- Apply **3 small perturbations** to this point
- Evolve all trajectories and visualize their divergence
- This demonstrates the concept behind the Lyapunov exponent calculation

In [ ]:
# ============================================================================
# VISUALIZE PERTURBATION EVOLUTION (1 point, 3 perturbations)
# ============================================================================

# Select a point from the NATURE trajectory (long_trj_for_selection)
desired_time_step = 1000
demo_point_idx = desired_time_step
demo_point = long_trj_for_selection[demo_point_idx]

print("="*70)
print("  Perturbation Evolution Visualization (NATURE)")
print("="*70)
print(f"\nSelected point from NATURE trajectory:")
print(f"  Time step: {demo_point_idx}")
print(f"  Position: [{demo_point[0]:7.3f}, {demo_point[1]:7.3f}, {demo_point[2]:7.3f}]")

# Number of perturbations for this visualization
n_demo_perturbations = 3
demo_n_steps = 3500  # Evolve for 3500 steps (35 seconds)

# Evolve unperturbed trajectory
trj_unperturbed_demo = produce_trj(demo_point, n_steps=demo_n_steps)
print(f"\n✓ Unperturbed trajectory evolved: {trj_unperturbed_demo.shape}")

# Evolve perturbed trajectories
trj_perturbed_demo = []
perturbations_demo = []
for i in range(n_demo_perturbations):
    # Generate random perturbation (same magnitude as in full calculation)
    pert = perturbation_std_LE * (np.random.rand(3) * 2 - 1)
    perturbations_demo.append(pert)
    
    # Perturbed initial condition
    u_perturbed = demo_point + pert
    
    # Evolve perturbed trajectory
    trj_pert = produce_trj(u_perturbed, n_steps=demo_n_steps)
    trj_perturbed_demo.append(trj_pert)
    
    print(f"✓ Perturbed trajectory {i+1} evolved: perturbation = [{pert[0]:.2e}, {pert[1]:.2e}, {pert[2]:.2e}]")

# Create 2D visualization with 2 panels: full view + zoomed view
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# The starting time index is where the demo point is located
start_time_idx = demo_point_idx

# Create time axis for demo trajectories (starting from the selected point)
time_axis_demo_plot = start_time_idx + np.arange(demo_n_steps + 1)

# Define colors for perturbed trajectories
colors_pert = ['red', 'orange', 'purple']

# ============================================================================
# LEFT PANEL: FULL VIEW
# ============================================================================

# Plot NATURE trajectory (X component - black line)
nature_x_component = long_trj_for_selection[:, 0]
ax1.plot(np.arange(len(nature_x_component)), nature_x_component, 'k-', linewidth=1.5, 
        label='NATURE trajectory (full)', alpha=0.6, zorder=5)

# Plot unperturbed trajectory (blue line, thicker)
ax1.plot(time_axis_demo_plot, trj_unperturbed_demo[:, 0], 
        linewidth=2.5, color='blue', label='Unperturbed', zorder=10)

# Plot initial point (blue dot)
ax1.scatter(start_time_idx, demo_point[0],
          color='blue', s=150, marker='o', edgecolors='darkblue', linewidths=2.5,
          label='Initial point', zorder=15)

# Plot perturbed trajectories
for i, trj_pert in enumerate(trj_perturbed_demo):
    ax1.plot(time_axis_demo_plot, trj_pert[:, 0],
            linewidth=1, color=colors_pert[i], alpha=0.9,
            label=f'Perturbed {i+1}', zorder=11+i)

# Add zero reference lines
ax1.axhline(0, color='red', linewidth=0.8, linestyle='--', alpha=0.7, zorder=1)
ax1.axvline(0, color='red', linewidth=0.8, linestyle='--', alpha=0.7, zorder=1)

ax1.set_xlabel('Time Step', fontsize=12)
ax1.set_ylabel('X Component', fontsize=12)
ax1.set_title('Full View: Perturbation Evolution (NATURE)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9, loc='best', ncol=2)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-50, len(nature_x_component) + 50)

# ============================================================================
# RIGHT PANEL: ZOOMED VIEW AROUND INITIAL POINT
# ============================================================================

# Define zoom window (show first ~600 steps from initial point)
zoom_start = start_time_idx - 20  # Start a bit before initial point
zoom_end = start_time_idx + 600   # Show first 600 steps after initial point

# Find the indices in the demo trajectories that correspond to this window
demo_zoom_start = 0
demo_zoom_end = min(620, demo_n_steps + 1)  # 600 steps + 20 buffer

# Plot NATURE trajectory (zoomed)
zoom_mask_start = max(0, zoom_start)
zoom_mask_end = min(len(nature_x_component), zoom_end)
ax2.plot(np.arange(zoom_mask_start, zoom_mask_end), 
         nature_x_component[zoom_mask_start:zoom_mask_end], 
         'k-', linewidth=1.5, label='NATURE trajectory', alpha=0.6, zorder=5)

# Plot unperturbed trajectory (zoomed)
ax2.plot(time_axis_demo_plot[demo_zoom_start:demo_zoom_end], 
         trj_unperturbed_demo[demo_zoom_start:demo_zoom_end, 0], 
        '--',linewidth=3, color='blue', label='Unperturbed', zorder=10)

# Plot initial point
ax2.scatter(start_time_idx, demo_point[0],
          color='blue', s=200, marker='o', edgecolors='darkblue', linewidths=3,
          label='Initial point', zorder=15)

# Plot perturbed trajectories (zoomed)
for i, trj_pert in enumerate(trj_perturbed_demo):
    ax2.plot(time_axis_demo_plot[demo_zoom_start:demo_zoom_end], 
             trj_pert[demo_zoom_start:demo_zoom_end, 0],
            linewidth=2, color=colors_pert[i], alpha=0.9,
            label=f'Perturbed {i+1}', zorder=11+i)

# Add zero reference line
ax2.axhline(0, color='red', linewidth=0.8, linestyle='--', alpha=0.7, zorder=1)

ax2.set_xlabel('Time Step', fontsize=12)
ax2.set_ylabel('X Component', fontsize=12)
ax2.set_title(f'Zoomed View: First 600 Steps\n(Initial divergence)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9, loc='best')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(zoom_start, zoom_end)

plt.tight_layout()
plt.savefig(f'{PathFigures}/perturbation_evolution_nature.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Figure saved: perturbation_evolution_nature.png")
print(f"\nVisualization shows (X component only):")
print(f"  LEFT PANEL (Full view):")
print(f"    - NATURE trajectory COMPLETE (black)")
print(f"    - Unperturbed trajectory from time step {int(start_time_idx)} (blue, thick)")
print(f"    - {n_demo_perturbations} perturbed trajectories (red, orange, purple)")
print(f"    - Evolution time: {demo_n_steps} steps = {demo_n_steps*dt:.1f} seconds")
print(f"  RIGHT PANEL (Zoomed view):")
print(f"    - Shows first 600 steps from initial point")
print(f"    - Reveals initial divergence of perturbed trajectories")
print(f"    - Time range: {zoom_start:.0f} to {zoom_end:.0f} steps")
print(f"\nNote: Perturbation magnitude = {perturbation_std_LE:.0e} (same as full calculation)")

In [ ]:
# ============================================================================
# COMPUTE ENSEMBLE DISTANCES (only if not loaded from cache)
# ============================================================================

if not skip_ensemble_computation:
    print("="*70)
    print(f"  Computing Ensemble Trajectories and Distances (NATURE)")
    print("="*70+"\n")

    # Storage for distances: (n_attractor_points, n_perturbations, n_steps)
    distances_ensemble = np.zeros((n_attractor_points_LE, n_perturbations_LE, n_steps_LE+1))

    # Process each attractor point
    for point_idx, attractor_point in enumerate(attractor_points_LE):
        
        # Evolve unperturbed trajectory from this attractor point
        trj_unperturbed = produce_trj(attractor_point, n_steps=n_steps_LE)
        
        # Generate and evolve perturbed trajectories
        for pert_idx in range(n_perturbations_LE):
            # Generate random perturbation
            perturbation = perturbation_std_LE * (np.random.rand(3) * 2 - 1)
            
            # Perturbed initial condition
            u_perturbed = attractor_point + perturbation
            
            # Evolve perturbed trajectory
            trj_perturbed = produce_trj(u_perturbed, n_steps=n_steps_LE)
            
            # Compute distance at each time step
            distances_ensemble[point_idx, pert_idx, :] = np.linalg.norm(
                trj_perturbed - trj_unperturbed, axis=1
            )
        
        # Progress update
        if (point_idx + 1) % 20 == 0 or (point_idx + 1) == n_attractor_points_LE:
            print(f"  Processed {point_idx+1}/{n_attractor_points_LE} attractor points...")

    print(f"\n✓ Ensemble computation complete")
    print(f"  distances_ensemble shape: {distances_ensemble.shape}")
    print(f"  (attractor_points, perturbations, time_steps) = ({n_attractor_points_LE}, {n_perturbations_LE}, {n_steps_LE+1})")
else:
    print("\n⏭️  Skipping ensemble computation (using cached results)")

In [ ]:
# ============================================================================
# SAVE ENSEMBLE DISTANCES TO CACHE (if newly computed)
# ============================================================================

if not skip_ensemble_computation:
    # Save distances_ensemble to file
    with open(cache_filename, 'wb') as f:
        pickle.dump(distances_ensemble, f)
    
    print("="*70)
    print("  ✓ ENSEMBLE DISTANCES SAVED TO CACHE")
    print("="*70)
    print(f"\n✓ File saved: {cache_filename}")
    print(f"  Size on disk: {distances_ensemble.nbytes / (1024**2):.1f} MB")
    print(f"  Shape: {distances_ensemble.shape}")
    print(f"\n  Next time you run this notebook, these results will be loaded")
    print(f"  automatically, saving ~24 minutes of computation time!")
else:
    print("\n⏭️  Skipping save (distances already loaded from cache)")

In [ ]:
# ============================================================================
# COMPUTE LYAPUNOV EXPONENT FROM ENSEMBLE MEAN
# ============================================================================

print(f"\n{'='*70}")
print(f"  Computing Lyapunov Exponent from Ensemble Mean Distance")
print(f"{'='*70}\n")

# Flatten distances: combine all attractor points and perturbations
# Shape: (n_attractor_points * n_perturbations, n_steps) = (1000, 2001)
distances_flat = distances_ensemble.reshape(-1, n_steps_LE+1)
print(f"Flattened distances shape: {distances_flat.shape}")

# Compute ensemble mean distance (average over all trajectories)
distance_mean_ensemble = np.mean(distances_flat, axis=0)
print(f"Ensemble mean distance shape: {distance_mean_ensemble.shape}")

# Create time axis (in seconds)
time_axis_LE = np.arange(n_steps_LE+1) * dt

# Take logarithm of ensemble mean distance
log_distance_mean = np.log(distance_mean_ensemble)

# Fit linear regression to log(distance) vs time (up to FIT_END)
time_fit = time_axis_LE[:FIT_END_LE]
log_distance_fit = log_distance_mean[:FIT_END_LE]

# Perform linear fit: log(distance) = λ * t + log_offset
p, cov = np.polyfit(time_fit, log_distance_fit, 1, cov=True)
lambda_ensemble = p[0]        # Slope = Lyapunov exponent
log_offset_ensemble = p[1]    # Intercept
lambda_std = np.sqrt(cov[0, 0])  # Standard deviation of λ

# Compute R² for goodness of fit
y_pred_fit = lambda_ensemble * time_fit + log_offset_ensemble
ss_res = np.sum((log_distance_fit - y_pred_fit)**2)
ss_tot = np.sum((log_distance_fit - np.mean(log_distance_fit))**2)
r_squared_ensemble = 1 - (ss_res / ss_tot)

print(f"\n{'='*70}")
print(f"  RESULTS: Ensemble-Based Lyapunov Exponent")
print(f"{'='*70}")
print(f"\n  λ_ensemble = {lambda_ensemble:.6f} ± {lambda_std:.6f}")
print(f"  Fit window: 0 to {FIT_END_LE} steps ({time_fit[-1]:.1f} seconds)")
print(f"  R² = {r_squared_ensemble:.6f}")
print(f"\n  Reference λ (Viswanath 1998) = 0.9056")
print(f"  Absolute error: {abs(lambda_ensemble - 0.9056):.6f}")
print(f"  Relative error: {abs(lambda_ensemble - 0.9056)/0.9056*100:.2f}%")
print(f"\n✓ Lyapunov exponent computed successfully")

In [ ]:
# ============================================================================
# FIGURE 13: ENSEMBLE MEAN DISTANCE EVOLUTION
# ============================================================================

# Compute ensemble mean distance (already computed, but let's be explicit)
mean_distance = distance_mean_ensemble

# Define fit window (in steps) - exclude first 100 transient points
FIT_START_ens = 100              # Start fit after transient phase
FIT_END_ens = min(2200, len(mean_distance))

# Create time axis in time steps
time_axis_steps_ens = np.arange(len(mean_distance))
x_fit_steps_ens = np.arange(FIT_START_ens, FIT_END_ens)

# Compute fitted line for visualization (using ensemble mean Lyapunov exponent)
# Fit excluding the first 100 transient points
time_pts_ens = time_axis_steps_ens * dt
lyap_ens_fit, log_offset_ens = np.polyfit(time_pts_ens[FIT_START_ens:FIT_END_ens], 
                                           np.log(mean_distance[FIT_START_ens:FIT_END_ens]), 1)
fitted_line_regression_ens = np.exp(log_offset_ens + lyap_ens_fit * time_pts_ens[FIT_START_ens:FIT_END_ens])

# Compute R² of fit
y_fit_ens = np.log(mean_distance[FIT_START_ens:FIT_END_ens])
y_pred_ens = log_offset_ens + lyap_ens_fit * time_pts_ens[FIT_START_ens:FIT_END_ens]
ss_res_ens = np.sum((y_fit_ens - y_pred_ens)**2)
ss_tot_ens = np.sum((y_fit_ens - np.mean(y_fit_ens))**2)
r_squared_ens = 1 - (ss_res_ens / ss_tot_ens)

# Create plot (single panel, matching plot_norm_distances_LLE_Lorenz63 style)
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

# Ensemble mean distance evolution (log scale) - orange line
ax.semilogy(time_axis_steps_ens, mean_distance, label='Ensemble Mean Distance', 
            color='orange', linewidth=2, alpha=0.8)

# Linear fit - red dashed line
ax.semilogy(x_fit_steps_ens, fitted_line_regression_ens, 
            label=f'Linear Fit: λ = {lyap_ens_fit:.4f} (R² = {r_squared_ens:.4f})', 
            color='red', linestyle='--', linewidth=2.5)

# Vertical lines marking fit window
ax.axvline(x=FIT_START_ens, color='green', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Fit window start (step {FIT_START_ens})')
ax.axvline(x=FIT_END_ens, color='gray', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Fit window end (step {FIT_END_ens})')

ax.set_title(f'Distance Evolution: ENSEMBLE (λ_theoretical = 0.9056)\nAttractor points: {n_attractor_points_LE} | Perturbations: {n_perturbations_LE}', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Distance (log scale)', fontsize=12)
ax.legend(fontsize=10, loc='best')

# Improved x-axis tick labels: every 200 steps, rotated, view up to 3200
ax.set_xticks(np.arange(0, 3400, 200))  # Every 200 steps up to 3200
ax.tick_params(axis='x', labelsize=10, rotation=45)  # Smaller font, rotated labels
ax.set_xlim(-50, 3200)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PathFigures}/fig13_ensemble_distance_n{n_attractor_points_LE}_npert{n_perturbations_LE}_std{perturbation_std_LE:.0e}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Figure 13 saved: fig13_ensemble_distance_n{n_attractor_points_LE}_npert{n_perturbations_LE}_std{perturbation_std_LE:.0e}.png")
print(f"\nEnsemble mean distance Lyapunov exponent (excluding first {FIT_START_ens} transient points):")
print(f"  λ = {lyap_ens_fit:.6f}")
print(f"  R² = {r_squared_ens:.6f}")
print(f"  Fit window: steps {FIT_START_ens} to {FIT_END_ens} ({FIT_START_ens * dt:.1f}s to {FIT_END_ens * dt:.1f}s)")
print(f"\nReference (Viswanath 1998): λ = 0.9056")
print(f"  Absolute error: {abs(lyap_ens_fit - 0.9056):.6f}")
print(f"  Relative error: {abs(lyap_ens_fit - 0.9056)/0.9056*100:.2f}%")
print(f"\nComparison:")
print(f"  Fit excluding transients: λ = {lyap_ens_fit:.6f}")
print(f"  Previous calculation (from step 0): λ = {lambda_ensemble:.6f}")

## Figure 13 with stepwise renormalization

This version uses the same NATURE attractor points and ensemble size as Figure 13, but renormalizes each perturbation after every RK4 step. The plotted curve is an equivalent distance reconstructed from the cumulative logarithmic growth, so it remains comparable with the non-renormalized distance plot while avoiding finite-amplitude saturation.

In [ ]:
# ============================================================================
# FIGURE 13B: ENSEMBLE MEAN DISTANCE EVOLUTION WITH RENORMALIZATION
# ============================================================================

epsilon_LE_renorm = perturbation_std_LE
renorm_seed_LE = 4242
cache_version_renorm_LE = 'nature_renorm_v1'
cache_filename_renorm = CLE_NATURE_DATA_DIR / (
    f'distances_ensemble_nature_renorm_n{n_attractor_points_LE}'
    f'_npert{n_perturbations_LE}_nsteps{n_steps_LE}'
    f'_std{perturbation_std_LE:.0e}.pkl'
)


def lorenz63_rhs_batch(states):
    states = np.asarray(states, dtype=float)
    x = states[..., 0]
    y = states[..., 1]
    z = states[..., 2]
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return np.stack([dx, dy, dz], axis=-1)


def rk4_step_batch_nature(states):
    states = np.asarray(states, dtype=float)
    k1 = lorenz63_rhs_batch(states)
    k2 = lorenz63_rhs_batch(states + 0.5 * dt * k1)
    k3 = lorenz63_rhs_batch(states + 0.5 * dt * k2)
    k4 = lorenz63_rhs_batch(states + dt * k3)
    return states + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


def compute_nature_renorm_result():
    try:
        with open(cache_filename_renorm, 'rb') as f:
            result = pickle.load(f)
        if (
            result.get('cache_version') == cache_version_renorm_LE
            and result.get('n_attractor_points_LE') == n_attractor_points_LE
            and result.get('n_perturbations_LE') == n_perturbations_LE
            and result.get('n_steps_LE') == n_steps_LE
            and result.get('epsilon_LE_renorm') == epsilon_LE_renorm
            and result.get('renorm_seed_LE') == renorm_seed_LE
            and np.array_equal(result.get('random_indices_LE'), random_indices_LE)
        ):
            print(f'Loaded cached renormalized NATURE result: {cache_filename_renorm}')
            return result
        print('Renormalized cache found, but metadata differs. Recomputing.')
    except FileNotFoundError:
        print(f'No renormalized cache found. Computing: {cache_filename_renorm}')

    rng = np.random.default_rng(renorm_seed_LE)
    cumulative_log_growth_sum = np.zeros(n_steps_LE, dtype=float)

    for point_idx, attractor_point in enumerate(attractor_points_LE):
        reference_state = np.asarray(attractor_point, dtype=float).copy()

        directions = rng.normal(size=(n_perturbations_LE, 3))
        direction_norms = np.clip(np.linalg.norm(directions, axis=1), 1e-300, None)
        perturbations = epsilon_LE_renorm * directions / direction_norms[:, None]
        states = reference_state[None, :] + perturbations

        for step in range(n_steps_LE):
            reference_next = rk4_step_batch_nature(reference_state[None, :])[0]
            states_next = rk4_step_batch_nature(states)
            deltas = states_next - reference_next[None, :]
            distances = np.linalg.norm(deltas, axis=1)
            safe_distances = np.clip(distances, 1e-300, None)

            cumulative_log_growth_sum[step] += float(np.mean(np.log(safe_distances / epsilon_LE_renorm)))
            states = reference_next[None, :] + epsilon_LE_renorm * deltas / safe_distances[:, None]
            reference_state = reference_next

        if (point_idx + 1) % 20 == 0 or (point_idx + 1) == n_attractor_points_LE:
            print(f'  Renormalized NATURE: processed {point_idx + 1}/{n_attractor_points_LE} attractor points')

    mean_log_growth_per_step = cumulative_log_growth_sum / n_attractor_points_LE
    running_log_growth = np.cumsum(mean_log_growth_per_step)
    distance_mean_equivalent = epsilon_LE_renorm * np.exp(np.concatenate(([0.0], running_log_growth)))
    time_axis_steps_renorm = np.arange(n_steps_LE + 1)
    time_axis_seconds_renorm = time_axis_steps_renorm * dt

    fit_start = 100
    fit_end = min(2200, len(distance_mean_equivalent))
    coeffs, cov = np.polyfit(
        time_axis_seconds_renorm[fit_start:fit_end],
        np.log(distance_mean_equivalent[fit_start:fit_end]),
        1,
        cov=True,
    )
    y_fit = np.log(distance_mean_equivalent[fit_start:fit_end])
    y_pred = coeffs[0] * time_axis_seconds_renorm[fit_start:fit_end] + coeffs[1]
    ss_res = np.sum((y_fit - y_pred) ** 2)
    ss_tot = np.sum((y_fit - np.mean(y_fit)) ** 2)
    r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    result = {
        'cache_version': cache_version_renorm_LE,
        'methodology': 'nature_lorenz_stepwise_renormalization_equivalent_distance',
        'distance_mean_equivalent': distance_mean_equivalent,
        'running_log_growth': running_log_growth,
        'mean_log_growth_per_step': mean_log_growth_per_step,
        'time_axis_steps': time_axis_steps_renorm,
        'time_axis_seconds': time_axis_seconds_renorm,
        'lambda': float(coeffs[0]),
        'lambda_std': float(np.sqrt(cov[0, 0])) if np.ndim(cov) == 2 else np.nan,
        'intercept': float(coeffs[1]),
        'r_squared': float(r_squared),
        'fit_start_steps': int(fit_start),
        'fit_end_steps': int(fit_end),
        'n_attractor_points_LE': int(n_attractor_points_LE),
        'n_perturbations_LE': int(n_perturbations_LE),
        'n_steps_LE': int(n_steps_LE),
        'epsilon_LE_renorm': float(epsilon_LE_renorm),
        'perturbation_std_LE': float(perturbation_std_LE),
        'renorm_seed_LE': int(renorm_seed_LE),
        'random_indices_LE': random_indices_LE.copy(),
        'renormalized': True,
    }

    with open(cache_filename_renorm, 'wb') as f:
        pickle.dump(result, f)
    print(f'Saved renormalized NATURE cache: {cache_filename_renorm}')
    return result


result_nature_renorm = compute_nature_renorm_result()

mean_distance_renorm = np.clip(result_nature_renorm['distance_mean_equivalent'], 1e-300, None)
time_axis_steps_renorm = result_nature_renorm['time_axis_steps']
time_axis_seconds_renorm = result_nature_renorm['time_axis_seconds']
FIT_START_renorm = result_nature_renorm['fit_start_steps']
FIT_END_renorm = result_nature_renorm['fit_end_steps']
x_fit_steps_renorm = np.arange(FIT_START_renorm, FIT_END_renorm)

lyap_renorm_fit, log_offset_renorm = np.polyfit(
    time_axis_seconds_renorm[FIT_START_renorm:FIT_END_renorm],
    np.log(mean_distance_renorm[FIT_START_renorm:FIT_END_renorm]),
    1,
)
fitted_line_renorm = np.exp(
    log_offset_renorm + lyap_renorm_fit * time_axis_seconds_renorm[FIT_START_renorm:FIT_END_renorm]
)

fig, ax = plt.subplots(1, 1, figsize=(14, 6))
ax.semilogy(
    time_axis_steps_renorm,
    mean_distance_renorm,
    label='Renormalized Ensemble Equivalent Distance',
    color='tab:blue',
    linewidth=2,
    alpha=0.85,
)
ax.semilogy(
    x_fit_steps_renorm,
    fitted_line_renorm,
    label=f'Linear Fit: lambda = {lyap_renorm_fit:.4f} (R^2 = {result_nature_renorm["r_squared"]:.4f})',
    color='red',
    linestyle='--',
    linewidth=2.5,
)
ax.axvline(x=FIT_START_renorm, color='green', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Fit window start (step {FIT_START_renorm})')
ax.axvline(x=FIT_END_renorm, color='gray', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Fit window end (step {FIT_END_renorm})')

ax.set_title(
    f'Distance Evolution: ENSEMBLE with Renormalization (lambda_theoretical = 0.9056)\n'
    f'Attractor points: {n_attractor_points_LE} | Perturbations: {n_perturbations_LE}',
    fontsize=14,
    fontweight='bold',
)
ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Equivalent Distance (log scale)', fontsize=12)
ax.legend(fontsize=10, loc='best')
ax.set_xticks(np.arange(0, 3400, 200))
ax.tick_params(axis='x', labelsize=10, rotation=45)
ax.set_xlim(-50, 3200)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig13_renorm_filename = (
    f'fig13_ensemble_distance_renorm_n{n_attractor_points_LE}'
    f'_npert{n_perturbations_LE}_std{perturbation_std_LE:.0e}.png'
)
plt.savefig(f'{PathFigures}/{fig13_renorm_filename}', dpi=300, bbox_inches='tight')
plt.show()

print(f'\nFigure 13 renormalized saved: {fig13_renorm_filename}')
print(f'\nRenormalized ensemble Lyapunov exponent (equivalent distance):')
print(f'  lambda = {lyap_renorm_fit:.6f}')
print(f'  R^2 = {result_nature_renorm["r_squared"]:.6f}')
print(f'  Fit window: steps {FIT_START_renorm} to {FIT_END_renorm} ({FIT_START_renorm * dt:.1f}s to {FIT_END_renorm * dt:.1f}s)')
print(f'  Reference (Viswanath 1998): lambda = 0.9056')
print(f'  Absolute error: {abs(lyap_renorm_fit - 0.9056):.6f}')
print(f'  Relative error: {abs(lyap_renorm_fit - 0.9056) / 0.9056 * 100:.2f}%')
